# Custom HuggingFace model with ms-swift LoRA SFT on Modal

Fine-tunes a **custom** HuggingFace model (`HuggingFaceTB/SmolLM2-135M`,
a 135M-parameter open Llama-family model) with LoRA SFT via ms-swift's
Megatron backend. 1 node × 1×H100.

The point of this tutorial is the *custom model* seam: the model is not
a member of the in-repo catalog. Instead we define a one-file subclass of
`ModelConfiguration` inline — `model_name`, optional `model_path`, and an
overridden `download_model()` is all that is needed.

Invoke any function on the returned `app` via `modal run`, or
interactively with `app.run()` + `.remote()`.

In [ ]:
%uv pip install -q git+https://github.com/modal-projects/training-gym.git@joy/initial-setup

In [ ]:
import modal

from modal_training_gym.common.dataset import DatasetConfig
from modal_training_gym.common.models import ModelConfiguration
from modal_training_gym.common.wandb import WandbConfig
from modal_training_gym.frameworks.ms_swift import (
    MsSwiftConfig,
    MsSwiftFrameworkConfig,
)
from modal_training_gym.frameworks.ms_swift.config import HF_CACHE_PATH

## Define the custom model

Subclass `ModelConfiguration` with the HuggingFace repo id as
`model_name` and a `download_model()` body that materializes the weights
into the shared HuggingFace cache. No enum tag, no registry entry, no
fork of the library — everything a framework needs to know is carried
on the subclass.

In [ ]:
class SmolLM2_135M(ModelConfiguration):
    model_name = "HuggingFaceTB/SmolLM2-135M"

    def download_model(self) -> None:
        from huggingface_hub import snapshot_download

        snapshot_download(repo_id=self.model_name)

## Define the dataset

ms-swift expects training data as a JSONL file with a `messages` field.
A tiny GSM8K slice keeps the Tier 2 smoke run cheap — we only need one
training step's worth of data to prove the custom-model seam works end
to end.

In [ ]:
class TinyGSM8KDataset(DatasetConfig):
    def __init__(
        self,
        hf_cache_root,
        data_folder="gsm8k_tiny",
        hf_dataset="openai/gsm8k",
        split="train[:4]",
        input_col="question",
        output_col="answer",
    ):
        self._hf_cache_root = str(hf_cache_root)
        self._data_folder = data_folder
        self._hf_dataset = hf_dataset
        self._split = split
        self._input_col = input_col
        self._output_col = output_col
        self.prompt_data = (
            f"{self._hf_cache_root}/msswift-data/{data_folder}/training.jsonl"
        )

    def prepare(self):
        import json
        import os

        from datasets import load_dataset

        output_dir = f"{self._hf_cache_root}/msswift-data/{self._data_folder}"
        os.makedirs(output_dir, exist_ok=True)

        try:
            ds = load_dataset(self._hf_dataset, split=self._split)
        except ValueError:
            ds = load_dataset(self._hf_dataset, "main", split=self._split)

        out_path = f"{output_dir}/training.jsonl"
        with open(out_path, "w") as f:
            for row in ds:
                messages = [
                    {"role": "user", "content": row[self._input_col]},
                    {"role": "assistant", "content": row[self._output_col]},
                ]
                f.write(json.dumps({"messages": messages}) + "\n")
        print(f"Wrote {out_path}")

## Define the experiment

Attach the custom `SmolLM2_135M` subclass to `MsSwiftConfig` just like a
built-in — the ms-swift launcher only reads `config.model.model_name`,
which our subclass provides. Tiny parallelism settings (single node,
single GPU) keep Tier 2 validation cheap.

In [ ]:
swift_framework_config = MsSwiftFrameworkConfig(
    # NGC PyTorch 25.01 ships Python 3.12 + PyTorch 2.6 + CUDA 12.6, which
    # matches the repo's pinned local Python (3.12) — Modal's
    # `serialized=True` requires the remote image's Python to match the
    # local one. The default ModelScope image pins Python 3.11 and would
    # trip `InvalidError` at app-build time.
    image="nvcr.io/nvidia/pytorch:25.01-py3",
    gpu="H100",
    n_nodes=1,
    gpus_per_node=1,
    tensor_model_parallel_size=1,
    expert_model_parallel_size=1,
    pipeline_model_parallel_size=1,
    context_parallel_size=1,
    sequence_parallel=False,
    num_train_epochs=1,
    lr=1e-4,
    global_batch_size=1,
    max_length=512,
    tuner_type="lora",
    lora_rank=8,
    lora_alpha=16,
    merge_lora=False,
    log_interval=1,
    save_interval=10,
    eval_iters=0,
)

my_training_run = MsSwiftConfig(
    dataset=TinyGSM8KDataset(HF_CACHE_PATH),
    model=SmolLM2_135M(),
    wandb=WandbConfig(project="custom-hf-smoke"),
    framework_config=swift_framework_config,
)

## Build the Modal app

In [ ]:
app = my_training_run.build_app()

## Run it

Interactive — open an ephemeral app and run one stage per cell:

In [ ]:
with modal.enable_output():
    with app.run():
        app.download_model.remote()

In [ ]:
with modal.enable_output():
    with app.run():
        app.prepare_dataset.remote()

In [ ]:
with modal.enable_output():
    with app.run():
        app.train.remote()